In [ ]:
import os
import sys

# Add parent directory to path for imports
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)  # Change working directory for relative paths

In [2]:
from src.docling_pipeline import create_converter
from pathlib import Path
from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import PdfFormatOption
from docling.datamodel.base_models import InputFormat

In [3]:
from src.bbox_draw import draw_bboxes_on_pdf

In [3]:
# pipeline_options = PdfPipelineOptions(
#     do_ocr=True,  # disable internal OCR
#     ocr_options=MMOcrOptions()
# )
# pipeline_options.ocr_options.lang = ["hi"]
# converter = DocumentConverter(
#     format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
# )

In [5]:
converter = create_converter("hin", 3)

In [6]:
d = converter.convert(Path("books/Class_6-Science-Hindi/Chapter_2.pdf"))

OSD failed (doc books/Class_6-Science-Hindi/Chapter_2.pdf, page: 0, OCR rectangle: 0, processed image file <tempfile._TemporaryFileWrapper object at 0x7f0a2473ebf0>):
 b'Estimating resolution as 1195\nToo few characters. Skipping this page\nWarning. Invalid resolution 0 dpi. Using 70 instead.\nToo few characters. Skipping this page\nError during processing.\n'


In [7]:
draw_bboxes_on_pdf(d.document, Path("books/Class_6-Science-Hindi/Chapter_2.pdf"))

Annotated PDF saved to: annotated_output.pdf


In [24]:
d.document.save_as_json("bruh.json")

In [12]:
import fitz


def extract_text_layer_only(input_pdf, output_pdf):
    """Keep text layer with original positioning, remove everything else."""

    doc = fitz.open(input_pdf)

    for page in doc:
        # Remove images
        for img in page.get_images():
            page.delete_image(img[0])

        # Remove drawings/vector graphics
        page.clean_contents()

        # This removes all non-text content while preserving text positioning
        page.wrap_contents()

    doc.save(output_pdf, garbage=4, deflate=True)
    print(f"Created: {output_pdf}")

In [15]:
extract_text_layer_only("./books/Class_6-Science-Hindi/Chapter_2.pdf", "text.pdf")

Created: text.pdf


In [16]:
d = converter.convert(Path("text.pdf"))

OSD failed (doc text.pdf, page: 0, OCR rectangle: 0, processed image file <tempfile._TemporaryFileWrapper object at 0x7f60aadcb6d0>):
 b'Estimating resolution as 653\nToo few characters. Skipping this page\nWarning. Invalid resolution 0 dpi. Using 70 instead.\nToo few characters. Skipping this page\nError during processing.\n'


In [17]:
draw_bboxes_on_pdf(d.document, Path("books/Class_6-Science-Hindi/Chapter_2.pdf"))

Annotated PDF saved to: annotated_output.pdf


In [35]:
def remove_watermark(input_pdf, output_pdf):
    """Keep text layer with original positioning, remove everything else."""

    doc = fitz.open(input_pdf)

    for page in doc:
        # Remove images
        for img in page.get_images():
            if img[5] == "DeviceGray":
                page.delete_image(img[0])

        # Remove drawings/vector graphics
        page.clean_contents()

        # This removes all non-text content while preserving text positioning
        page.wrap_contents()

    doc.save(output_pdf, garbage=4, deflate=True)
    print(f"Created: {output_pdf}")

In [36]:
remove_watermark("./books/Class_6-Science-Hindi/Chapter_2.pdf", "clean.pdf")

Created: clean.pdf


In [37]:
d = converter.convert(Path("clean.pdf"))
draw_bboxes_on_pdf(
    d.document, Path("books/Class_6-Science-Hindi/Chapter_2.pdf"), "clean_ocr.pdf"
)

OSD failed (doc clean.pdf, page: 0, OCR rectangle: 0, processed image file <tempfile._TemporaryFileWrapper object at 0x7f60245c3d90>):
 b'Estimating resolution as 653\nToo few characters. Skipping this page\nWarning. Invalid resolution 0 dpi. Using 70 instead.\nToo few characters. Skipping this page\nError during processing.\n'


Annotated PDF saved to: clean_ocr.pdf
